# Sprint 2: Spatial Questions to Market Data

**GeoRAD Emerging Spatial Professionals Mentorship Program · Week 3**

This notebook helps me:

1. Upload and **inspect** the five-town GeoJSON from my Sprint 1
2. It will Extract town names as workflow inputs
3. Run an **AI-assisted Google search** (via SerpApi Google AI Overview) for current land price signals
4. Build a structured **market dataset** for the next sprint

I don't have  to build an AI model. The focus is  on understanding the workflow, running it, inspecting results, and documenting what you find.

> Keep your SerpApi key private. Do not commit this notebook or your key to a public GitHub repo.


In [ ]:
## 0. Mount Drive

In [2]:
# ── Cell 1: Mount and navigate ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/GEORAD_SPRINT_2')
print("📁 Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Working directory: /content/drive/MyDrive/GEORAD_SPRINT_2


## 1. Install packages

Run once per Colab session.


In [3]:
# SerpApi client + helpers for tables / files
%pip install -q google-search-results pandas


  Preparing metadata (setup.py) ... done


## 2. Configure your SerpApi key

1. Create a key at [serpapi.com](https://serpapi.com/)
2. Prefer Colab Secrets (`SERPAPI_API_KEY`) or paste when prompted

Never hard-code the key into a cell you will share publicly.


In [5]:
import os
from getpass import getpass

try:
    from google.colab import userdata  # type: ignore
    SERPAPI_API_KEY = userdata.get("secretName")
except Exception:
    SERPAPI_API_KEY = None

if not SERPAPI_API_KEY:
    SERPAPI_API_KEY = os.environ.get("SERPAPI_API_KEY") or getpass("SerpApi API key: ").strip()

assert SERPAPI_API_KEY, "SerpApi API key is required."
print("SerpApi key loaded.")


SerpApi API key: ··········
SerpApi key loaded.


## 3. Upload your towns GeoJSON

Upload the **selected towns** GeoJSON from Sprint 1 (five towns inside your assigned LCDA).

GeoRAD town exports typically look like:

```json
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "ward": "Idiya",
        "lga": "Abeokuta North",
        "state": "Ogun"
      },
      "geometry": { "type": "MultiPolygon", "coordinates": [] }
    }
  ]
}
```

**Town name field:** use `ward` (primary). The notebook will detect this automatically.


In [8]:
from pathlib import Path
import json

GEOJSON_PATH = Path(
    "/content/drive/MyDrive/GEORAD_SPRINT_2/bariga_selected_wards.geojson"
)

if not GEOJSON_PATH.exists():
    raise FileNotFoundError(f"GeoJSON file not found: {GEOJSON_PATH}")

with GEOJSON_PATH.open("r", encoding="utf-8") as f:
    geojson = json.load(f)

print(f"✅ Loaded: {GEOJSON_PATH.name}")
print(f"📦 Size: {GEOJSON_PATH.stat().st_size:,} bytes")

✅ Loaded: bariga_selected_wards.geojson
📦 Size: 21,628 bytes


## 4. Inspect the GeoJSON

Before searching the market, confirm the file is valid and that the **ward** names look correct.


In [15]:
from collections import Counter

print("=== GeoJSON Inspection ===")

# ── Top-level information ────────────────────────────────
print("\n=== Top-level ===")
print("Type:", geojson.get("type"))
print("Name:", geojson.get("name"))
print("CRS :", geojson.get("crs"))
print("Keys:", sorted(geojson.keys()))

# ── Get features ─────────────────────────────────────────
features = geojson.get("features")

if not isinstance(features, list):
    raise TypeError("Expected a GeoJSON FeatureCollection with a features array.")

print("\n=== Features ===")
print("Count:", len(features))

# Geometry types
geom_types = Counter(
    (feature.get("geometry") or {}).get("type", "missing")
    for feature in features
)

print("Geometry types:", dict(geom_types))

# Property keys
property_keys = Counter()

for feature in features:
    properties = feature.get("properties") or {}

    if isinstance(properties, dict):
        property_keys.update(properties.keys())

print("Property keys:", dict(property_keys))


TOWN_NAME_FIELD = "wardname"

print("\nUsing name field:", TOWN_NAME_FIELD)

# Check field actually exists
missing_names = []

for i, feature in enumerate(features, start=1):
    properties = feature.get("properties") or {}

    if not properties.get(TOWN_NAME_FIELD):
        missing_names.append(i)

if missing_names:
    print(
        f"WARNING: {TOWN_NAME_FIELD!r} is missing/empty "
        f"for feature(s): {missing_names}"
    )

# ── Feature preview ──────────────────────────────────────
print("\n=== Feature Preview ===")

for i, feature in enumerate(features, start=1):

    properties = feature.get("properties") or {}
    geometry = feature.get("geometry") or {}

    ward = properties.get("wardname", "(unnamed)")
    lga = properties.get("lga")
    state = properties.get("state")

    geometry_type = geometry.get("type", "missing")

    extra = []

    if lga:
        extra.append(f"lga={lga}")

    if state:
        extra.append(f"state={state}")

    suffix = f" | {', '.join(extra)}" if extra else ""

    print(
        f"{i}. {ward} | "
        f"{geometry_type}"
        f"{suffix}"
    )

# ── Expected feature count ───────────────────────────────
if len(features) != 5:
    print(
        f"\n⚠️ WARNING: Expected 5 features, "
        f"but found {len(features)}."
    )
else:
    print("\n✅ Found 5 features.")

=== GeoJSON Inspection ===

=== Top-level ===
Type: FeatureCollection
Name: bariga_selected_wards
CRS : {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}
Keys: ['crs', 'features', 'name', 'type']

=== Features ===
Count: 5
Geometry types: {'MultiPolygon': 5}
Property keys: {'FID': 5, 'globalid': 5, 'uniq_id': 5, 'timestamp': 5, 'editor': 5, 'wardname': 5, 'wardcode': 5, 'lganame': 5, 'lgacode': 5, 'statename': 5, 'statecode': 5, 'amapcode': 5, 'status': 5, 'source': 5, 'urban': 5}

Using name field: wardname

=== Feature Preview ===
1. Ibuowo / Owotutu | MultiPolygon
2. Ilaje | MultiPolygon
3. Pedro / Gbagada | MultiPolygon
4. Aiyetoro / Mafowoku | MultiPolygon
5. Owode  / Orile Bariga | MultiPolygon

✅ Found 5 features.


## 5. Extract the five town names

Uses the detected property (usually **`ward`**) as the search input.


In [16]:
# ward/town extraction names from the GeoJSON

towns = [
    feature.get("properties", {}).get("wardname")
    for feature in features
]

# Empty values removed
towns = [town.strip() for town in towns if town and town.strip()]

# Duplicates while preserving order
towns = list(dict.fromkeys(towns))

print("Towns to search:")

for i, town in enumerate(towns, start=1):
    print(f"  {i}. {town}")

if not towns:
    raise ValueError("No town names found in the GeoJSON.")

print(f"\n✅ {len(towns)} towns loaded.")

Towns to search:
  1. Ibuowo / Owotutu
  2. Ilaje
  3. Pedro / Gbagada
  4. Aiyetoro / Mafowoku
  5. Owode  / Orile Bariga

✅ 5 towns loaded.


## 6. Market question template

Starting question (edit if you want to experiment later):

> What is the current land price per square metre in [Town], Lagos?


In [20]:

# Optional override for experimentation:
QUESTION_TEMPLATE = "Current asking price of residential land per sqm in {town}, Shomolu Lagos Nigeria"

for town in towns:
    print(QUESTION_TEMPLATE.format(town=town))


Current asking price of residential land per sqm in Ibuowo / Owotutu, Shomolu Lagos Nigeria
Current asking price of residential land per sqm in Ilaje, Shomolu Lagos Nigeria
Current asking price of residential land per sqm in Pedro / Gbagada, Shomolu Lagos Nigeria
Current asking price of residential land per sqm in Aiyetoro / Mafowoku, Shomolu Lagos Nigeria
Current asking price of residential land per sqm in Owode  / Orile Bariga, Shomolu Lagos Nigeria


## 7. Fetch Google AI Overview via SerpApi

Flow:

1. Google Search (`engine=google`) for the question (`hl=en`, `gl=ng`)
2. If AI Overview returns a `page_token`, call `engine=google_ai_overview` immediately
3. Collect overview text + first supporting reference

Run towns **sequentially**. AI Overview tokens expire quickly.


In [21]:
from datetime import date
import re
import time
from serpapi import GoogleSearch


def flatten_text_blocks(blocks) -> str:
    parts = []
    if not isinstance(blocks, list):
        return ""
    for block in blocks:
        if not isinstance(block, dict):
            continue
        if block.get("snippet"):
            parts.append(str(block["snippet"]))
        if block.get("list"):
            for item in block["list"]:
                if isinstance(item, dict) and item.get("snippet"):
                    parts.append(str(item["snippet"]))
                elif isinstance(item, str):
                    parts.append(item)
        if block.get("text_blocks"):
            parts.append(flatten_text_blocks(block["text_blocks"]))
    return "\n".join(p for p in parts if p)


def fetch_ai_overview(query: str) -> dict:
    search = GoogleSearch(
        {
            "engine": "google",
            "q": query,
            "hl": "en",
            "gl": "ng",
            "api_key": SERPAPI_API_KEY,
            "no_cache": "true",
        }
    )
    results = search.get_dict()
    if results.get("error"):
        return {"error": results["error"], "query": query}

    ai = results.get("ai_overview") or {}
    if ai.get("page_token"):
        follow = GoogleSearch(
            {
                "engine": "google_ai_overview",
                "page_token": ai["page_token"],
                "api_key": SERPAPI_API_KEY,
                "no_cache": "true",
            }
        )
        follow_results = follow.get_dict()
        if follow_results.get("error"):
            return {"error": follow_results["error"], "query": query, "stage": "ai_overview"}
        ai = follow_results.get("ai_overview") or follow_results

    text = flatten_text_blocks(ai.get("text_blocks"))
    if not text and ai.get("snippet"):
        text = str(ai["snippet"])

    references = ai.get("references") or []
    return {
        "query": query,
        "text": text.strip(),
        "references": references if isinstance(references, list) else [],
        "has_ai_overview": bool(text or references),
        "raw_keys": sorted(ai.keys()) if isinstance(ai, dict) else [],
    }


def extract_price_ngn_per_sqm(text: str):
    if not text:
        return None
    patterns = [
        r"₦\s*([\d,]+(?:\.\d+)?)\s*(?:per|/)?\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"NGN\s*([\d,]+(?:\.\d+)?)\s*(?:per|/)?\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"([\d,]+(?:\.\d+)?)\s*(?:naira|NGN|₦)\s*(?:per|/)\s*(?:sq\.?\s*m|square\s*met(?:re|er))",
        r"([\d,]+(?:\.\d+)?)\s*/\s*(?:sqm|sq\.?\s*m)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return f"₦{match.group(1).replace(',', '')}/sqm"
    match = re.search(r"₦\s*([\d,]+(?:\.\d+)?)", text)
    if match:
        return f"₦{match.group(1).replace(',', '')} (unit unclear; verify)"
    return None


def first_source(references: list) -> str:
    if not references:
        return ""
    ref = references[0] if isinstance(references[0], dict) else {}
    title = str(ref.get("title") or "").strip()
    link = str(ref.get("link") or ref.get("source") or "").strip()
    if title and link:
        return f"{title} | {link}"
    return link or title or ""


search_date = date.today().isoformat()
overview_cache = {}

for town in towns:
    query = QUESTION_TEMPLATE.format(town=town)
    print(f"\n=== {town} ===")
    print("Query:", query)
    result = fetch_ai_overview(query)
    overview_cache[town] = result
    if result.get("error"):
        print("ERROR:", result["error"])
    else:
        print("AI Overview found:", result.get("has_ai_overview"))
        preview = (result.get("text") or "")[:400]
        suffix = "..." if len(result.get("text") or "") > 400 else ""
        print("Text preview:", preview + suffix)
        print("References:", len(result.get("references") or []))
    time.sleep(1.5)

print("\nDone fetching AI Overviews.")



=== Ibuowo / Owotutu ===
Query: Current asking price of residential land per sqm in Ibuowo / Owotutu, Shomolu Lagos Nigeria
AI Overview found: True
Text preview: The current asking price of residential land in the Ibuowo / Owotutu area of Bariga, Shomolu, Lagos ranges between ₦250,000 and ₦415,000 per square meter.
Because land in this developed tier of Lagos Mainland is rarely sold as empty, bare earth, pricing is heavily determined by total plot sizes and the structures currently built on them (often sold as "demolishable structures").
Price Breakdown by...
References: 7

=== Ilaje ===
Query: Current asking price of residential land per sqm in Ilaje, Shomolu Lagos Nigeria
AI Overview found: True
Text preview: The current asking price for residential and mixed-use land around the Ilaje Road area in Bariga/Shomolu, Lagos, ranges from approximately ₦200,000 to ₦320,000 per square meter (sqm) based on active local listings.
Price Breakdown by Listing Samples
Main Road / High-Traffic Plo

## 8. Build the Sprint 2 market dataset

Required columns:

| Field | Description |
| --- | --- |
| Town | Assigned town name |
| Current Land Price/sqm | Indicative asking-price level (₦/sqm) |
| Supporting Source | One supporting source/listing |
| Date Searched | Search date |


In [22]:
import pandas as pd

records = []
for town in towns:
    result = overview_cache.get(town) or {}
    text = result.get("text") or ""
    price = extract_price_ngn_per_sqm(text) if not result.get("error") else None
    source = first_source(result.get("references") or [])
    if not source and text:
        source = "AI Overview text only (no structured reference returned)"
    if result.get("error"):
        price = price or "ERROR"
        source = result["error"]
    elif not price:
        price = "Not found / needs manual review"

    records.append(
        {
            "Town": town,
            "Current Land Price/sqm": price,
            "Supporting Source": source,
            "Date Searched": search_date,
            "Query": result.get("query", QUESTION_TEMPLATE.format(town=town)),
            "Overview excerpt": (text[:280] + "...") if len(text) > 280 else text,
        }
    )

market_df = pd.DataFrame(records)
display_cols = ["Town", "Current Land Price/sqm", "Supporting Source", "Date Searched"]
market_df[display_cols]


,Town,Current Land Price/sqm,Supporting Source,Date Searched
0,Ibuowo / Owotutu,₦415000/sqm,"Land for Sale in Shomolu, Lagos - Nigeria Prop...",2026-09-26
1,Ilaje,₦320000/sqm,"Land for Sale in Bariga, Shomolu, Lagos | http...",2026-09-26
2,Pedro / Gbagada,₦1200000/sqm,FOR SALE | DIRECT SALES A prime 725sqm parcel ...,2026-09-26
3,Aiyetoro / Mafowoku,₦420000/sqm,"Land for Sale in Shomolu, Lagos | https://nige...",2026-09-26
4,Owode / Orile Bariga,₦250000/sqm,"Residential Land for Sale in Bariga, Shomolu, ...",2026-09-26


## 9. Inspect results (do not stop at run)

Check:

- Were all five towns processed?
- Did each town return a market price?
- Does the result look plausible for Lagos land markets?
- Was supporting evidence returned?
- Did anything fail or look unexpected?

Then experiment by changing `QUESTION_TEMPLATE` and re-running sections 5–7.


In [23]:
print("Towns processed:", len(market_df))
print("Price values:")
print(market_df["Current Land Price/sqm"].value_counts(dropna=False))
print("\nRows needing manual review:")
needs_review = market_df[
    market_df["Current Land Price/sqm"]
    .astype(str)
    .str.contains("Not found|ERROR|unclear|verify", case=False, na=False)
]
needs_review[display_cols] if len(needs_review) else "None flagged by heuristics."


Towns processed: 5
Price values:
Current Land Price/sqm
₦415000/sqm     1
₦320000/sqm     1
₦1200000/sqm    1
₦420000/sqm     1
₦250000/sqm     1
Name: count, dtype: int64

Rows needing manual review:


'None flagged by heuristics.'

In [24]:
TOWN_TO_INSPECT = towns[3]
detail = overview_cache.get(TOWN_TO_INSPECT) or {}
print("Town:", TOWN_TO_INSPECT)
print("Has overview:", detail.get("has_ai_overview"))
print("Raw AI keys:", detail.get("raw_keys"))
print("\nFull text:\n")
print(detail.get("text") or "(empty)")
print("\nReferences:")
for i, ref in enumerate(detail.get("references") or [], start=1):
    if isinstance(ref, dict):
        print(f"{i}. {ref.get('title')} -> {ref.get('link')}")
    else:
        print(f"{i}. {ref}")


Town: Aiyetoro / Mafowoku
Has overview: True
Raw AI keys: ['references', 'text_blocks']

Full text:

The current asking price of residential land in the Aiyetoro and Mafowoku areas of Shomolu, Lagos ranges between ₦150,000 and ₦420,000 per square meter (sqm).
Because Shomolu is a highly developed, high-density metropolitan hub, raw "bare land" is exceptionally rare. Most land sales in these neighborhoods consist of plots containing older, demolishable bungalows or multi-story buildings being sold primarily for their footprint value.
Price Breakdown by Plot Type
Plots in this region are typically valued by total plot size rather than raw square meters, converting roughly to these current benchmarks:
Per Square Meter (sqm): ₦150,000 – ₦420,000 / sqm
Half Plot (~300 sqm): ₦70,000,000 – ₦110,000,000
Full Plot (~600 sqm): ₦150,000,000 – ₦250,000,000
Key Pricing Factors
Road Proximity & Accessibility: Plots closer to major conduits like Fola Agoro, Bajulaiye Road, or the Ikorodu Road axis co

## 10. Export your market dataset

Download these files for Submission 2 (dataset + repo).


In [25]:
from pathlib import Path

out_dir = Path("sprint2_outputs")
out_dir.mkdir(exist_ok=True)

csv_path = out_dir / "market_dataset.csv"
json_path = out_dir / "market_dataset.json"
overviews_path = out_dir / "ai_overview_raw.json"

export_df = market_df[display_cols].copy()
export_df.to_csv(csv_path, index=False)
export_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

serialisable = {}
for town, payload in overview_cache.items():
    serialisable[town] = {
        "query": payload.get("query"),
        "error": payload.get("error"),
        "text": payload.get("text"),
        "references": payload.get("references"),
        "has_ai_overview": payload.get("has_ai_overview"),
    }
overviews_path.write_text(json.dumps(serialisable, ensure_ascii=False, indent=2), encoding="utf-8")

print("Wrote:", csv_path.resolve())
print("Wrote:", json_path.resolve())
print("Wrote:", overviews_path.resolve())

try:
    from google.colab import files  # type: ignore
    files.download(str(csv_path))
    files.download(str(json_path))
except Exception:
    print("Not in Colab download mode; files are on disk in sprint2_outputs/.")


Wrote: /content/drive/MyDrive/GEORAD_SPRINT_2/sprint2_outputs/market_dataset.csv
Wrote: /content/drive/MyDrive/GEORAD_SPRINT_2/sprint2_outputs/market_dataset.json
Wrote: /content/drive/MyDrive/GEORAD_SPRINT_2/sprint2_outputs/ai_overview_raw.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Optional experiments

1. Change the question wording and compare prices/sources
2. Add "residential" or "commercial" to the prompt
3. Test one extra nearby town (not required for submission)
4. If a town fails, use an AI assistant to interpret the error or overview text

Goal: understand what the technology can do for a spatial market question.


In [ ]:
# Example: re-run a single town with an alternate prompt
# alt_query = f"Residential land asking price per square metre in {towns[0]}, Lagos Nigeria"
# alt = fetch_ai_overview(alt_query)
# print(alt.get("text") or alt.get("error"))
# print(extract_price_ngn_per_sqm(alt.get("text") or ""))
